# --- TRAINING ---

In [ ]:
from google.colab import drive
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import numpy as np
import torchvision.transforms as T
import torchvision.models as models
import torch.nn.functional as F
import tqdm


In [ ]:
# Montage du Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# --- CONFIGURATION DES CHEMINS ---
# Dossiers TRAIN :
IMG_TRAIN = "/content/drive/MyDrive/bdd10k/img/train"
MASK_TRAIN = "/content/drive/MyDrive/bdd10k/labels/train"

# Dossiers VAL :
IMG_VAL = "/content/drive/MyDrive/bdd10k/img/val"
MASK_VAL = "/content/drive/MyDrive/bdd10k/labels/val"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Création de l'arborescence locale
!mkdir -p /content/local_bdd/img/train /content/local_bdd/img/val
!mkdir -p /content/local_bdd/labels/train /content/local_bdd/labels/val

#Transférer les données du drive en local sur colab
print("Copie des images (Train + Val)...")
!cp -r /content/drive/MyDrive/bdd10k/img/train/* /content/local_bdd/img/train/
!cp -r /content/drive/MyDrive/bdd10k/img/val/* /content/local_bdd/img/val/

print("Copie des masques (Train + Val)...")
!cp -r /content/drive/MyDrive/bdd10k/labels/train/* /content/local_bdd/labels/train/
!cp -r /content/drive/MyDrive/bdd10k/labels/val/* /content/local_bdd/labels/val/

print("Transfert terminé ! Tes données sont maintenant sur le SSD local.")

📦 Copie des images (Train + Val)...
🎭 Copie des masques (Train + Val)...
✅ Transfert terminé ! Tes données sont maintenant sur le SSD local.


In [ ]:
IMG_TRAIN = "/content/local_bdd/img/train"
MASK_TRAIN = "/content/local_bdd/labels/train"

IMG_VAL = "/content/local_bdd/img/val"
MASK_VAL = "/content/local_bdd/labels/val"

In [ ]:
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        # Upsample using ConvTranspose
        self.up = nn.ConvTranspose2d(in_channels, in_channels // 2, kernel_size=2, stride=2)

        # After concatenation, input channels = (upsampled_channels + skip_channels)
        combined_channels = (in_channels // 2) + skip_channels

        self.conv = nn.Sequential(
            nn.Conv2d(combined_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True)
        )

    def forward(self, x, skip):
        x = self.up(x)
        # MobileNetV2 usually matches dimensions, but we cat anyway
        x = torch.cat([x, skip], dim=1)
        return self.conv(x)


In [ ]:
class MobileNetV2_UNet(nn.Module):
    def __init__(self, n_class):
        super().__init__()

        # 1. Load Pre-trained Backbone
        backbone = models.mobilenet_v2(weights='IMAGENET1K_V1').features

        # 2. Encoder: Extract specific layers for skip connections
        self.layer0 = backbone[0:2]   # Output: 16 channels, Size: 1/2 (112x112)
        self.layer1 = backbone[2:4]   # Output: 24 channels, Size: 1/4 (56x56)
        self.layer2 = backbone[4:7]   # Output: 32 channels, Size: 1/8 (28x28)
        self.layer3 = backbone[7:14]  # Output: 96 channels, Size: 1/16 (14x14)
        self.layer4 = backbone[14:18] # Output: 320 channels, Bottleneck, Size: 1/32 (7x7)

        # 3. Decoder: Defined to match the asymmetric channels of MobileNetV2
        # DecoderBlock(in_channels, skip_channels, out_channels)
        self.up1 = DecoderBlock(320, 96, 128) # Input 320, Skip 96 -> 128
        self.up2 = DecoderBlock(128, 32, 64)  # Input 128, Skip 32 -> 64
        self.up3 = DecoderBlock(64, 24, 32)   # Input 64, Skip 24 -> 32
        self.up4 = DecoderBlock(32, 16, 16)   # Input 32, Skip 16 -> 16

        # 4. Final Head: Map to number of classes
        # We need one last upsample to get from 112x112 back to original 224x224
        self.out_up = nn.ConvTranspose2d(16, 16, kernel_size=2, stride=2)
        self.final_conv = nn.Conv2d(16, n_class, kernel_size=1)

    def forward(self, x):
        # Encoder path
        s0 = self.layer0(x)     # 16 channels
        s1 = self.layer1(s0)    # 24 channels
        s2 = self.layer2(s1)    # 32 channels
        s3 = self.layer3(s2)    # 96 channels
        x = self.layer4(s3)     # 320 channels (Bottleneck)

        # Decoder path with skip connections
        x = self.up1(x, s3)
        x = self.up2(x, s2)
        x = self.up3(x, s1)
        x = self.up4(x, s0)

        # Final upsampling to original input size
        x = self.out_up(x)
        return self.final_conv(x)


In [ ]:
class BDD10kDataset(Dataset):
    def __init__(self, img_dir, mask_dir, size=(384, 640)):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.size = size # Format (H, W)

        # On crée l'intersection pour éviter les erreurs
        raw_imgs = sorted([f for f in os.listdir(img_dir) if f.endswith('.jpg')])
        raw_masks = {f.replace('_train_id.png', ''): f for f in os.listdir(mask_dir) if f.endswith('.png')}

        self.valid_pairs = []
        for img_file in raw_imgs:
            img_id = img_file.split('.')[0]
            if img_id in raw_masks:
                self.valid_pairs.append((img_file, raw_masks[img_id]))

        # Transformation standard ImageNet pour MobileNetV2
        self.img_transform = T.Compose([
            T.Resize(self.size),
            T.ToTensor(),
            T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        ])

        self.mask_transform = T.Compose([
            T.Resize(self.size, interpolation=T.InterpolationMode.NEAREST),
            T.ToTensor()
        ])

    def __len__(self):
        return len(self.valid_pairs)

    def __getitem__(self, idx):
        img_name, mask_name = self.valid_pairs[idx]

        # Chargement
        image = Image.open(os.path.join(self.img_dir, img_name)).convert("RGB")
        mask = Image.open(os.path.join(self.mask_dir, mask_name))

        # Extraction de la ROUTE (ID 0 dans les labels sémantiques)
        mask_np = np.array(mask)
        road_mask = np.where(mask_np == 0, 1.0, 0.0).astype(np.float32)
        road_mask = Image.fromarray(road_mask)

        return self.img_transform(image), self.mask_transform(road_mask)

In [ ]:
train_dataset = BDD10kDataset(IMG_TRAIN, MASK_TRAIN)
val_dataset = BDD10kDataset(IMG_VAL, MASK_VAL)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False)

In [ ]:
# 2. Instancier ton modèle (n_class=1 pour la route)
model = MobileNetV2_UNet(n_class=1).to(DEVICE)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 139MB/s]


In [ ]:

# 3. Fonction de perte et Optimiseur
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
def train_fn(model, loader, optimizer, criterion):
    model.train()
    loop = tqdm.tqdm(loader)
    total_loss = 0

    for images, masks in loop:
        images, masks = images.to(DEVICE), masks.to(DEVICE)

        outputs = model(images)
        loss = criterion(outputs, masks)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    return total_loss / len(loader)

def validate_fn(model, loader, criterion):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(DEVICE), masks.to(DEVICE)
            outputs = model(images)
            loss = criterion(outputs, masks)
            total_loss += loss.item()
    return total_loss / len(loader)


In [ ]:
torch.cuda.empty_cache()
# --- LANCEMENT DE L'ENTRAÎNEMENT ---
EPOCHS = 30
for epoch in range(EPOCHS):
    train_loss = train_fn(model, train_loader, optimizer, criterion)
    val_loss = validate_fn(model, val_loader, criterion)

    print(f"Epoch [{epoch+1}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    # Sauvegarde du meilleur modèle
    if epoch % 5 == 0:
        torch.save(model.state_dict(), f"tesla_model_epoch_{epoch}.pth")

100%|██████████| 438/438 [05:48<00:00,  1.26it/s, loss=0.257]


Epoch [1/30] | Train Loss: 0.4531 | Val Loss: 0.2618


100%|██████████| 438/438 [05:54<00:00,  1.23it/s, loss=0.124]


Epoch [2/30] | Train Loss: 0.1918 | Val Loss: 0.1216


100%|██████████| 438/438 [05:50<00:00,  1.25it/s, loss=0.0751]


...

Epoch [11/30] | Train Loss: 0.0488 | Val Loss: 0.0892


100%|██████████| 438/438 [05:54<00:00,  1.24it/s, loss=0.0399]


Epoch [12/30] | Train Loss: 0.0474 | Val Loss: 0.0917


 10%|█         | 45/438 [00:36<05:19,  1.23it/s, loss=0.0791]


KeyboardInterrupt: 

# --- Inference ---

In [ ]:
# 1. Recréer l'architecture
model = MobileNetV2_UNet(n_class=1).to(DEVICE)

# 2. Charger les poids
model.load_state_dict(torch.load('model_epoch_5.pth', map_location=DEVICE))
model.eval() # Mode évaluation
print("Modèle chargé et prêt pour l'inférence.")

🚀 Modèle Tesla chargé et prêt pour l'inférence.


In [ ]:
import cv2
import torch
import numpy as np
from PIL import Image
import torchvision.transforms as T

import time

def process_video(input_path, output_path, model):
    model.eval()
    cap = cv2.VideoCapture(input_path)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_video = int(cap.get(cv2.CAP_PROP_FPS))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps_video, (width, height))

    transform = T.Compose([
        T.Resize((384, 640)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print("🚗 Traitement en cours...")

    while cap.isOpened():
        start_time = time.time() # Début du chrono

        ret, frame = cap.read()
        if not ret: break

        # 1. Inférence
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(img_rgb)
        input_tensor = transform(pil_img).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            output = model(input_tensor)
            pred = torch.sigmoid(output).cpu().numpy()[0][0]
            mask = (pred > 0.5).astype(np.uint8)

        # 2. Post-processing & Overlay
        mask_hd = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)
        blue_carpet = frame.copy()
        blue_carpet[mask_hd == 1] = [255, 100, 0]
        result_frame = cv2.addWeighted(blue_carpet, 0.4, frame, 0.6, 0)

        # 3. Calcul des FPS
        end_time = time.time()
        fps_current = 1 / (end_time - start_time)

        # 4. Affichage du texte FPS sur la frame
        cv2.putText(result_frame, f"FPS: {fps_current:.1f}", (50, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        out.write(result_frame)

    cap.release()
    out.release()
    print(f"✅ Terminé ! Vidéo sauvegardée sous : {output_path}")

process_video("video2.mp4", "resultat_tesla2.mp4", model)

🚗 Traitement en cours...
✅ Terminé ! Vidéo sauvegardée sous : resultat_tesla2.mp4


# --- YOLO ---

In [ ]:
!pip install ultralytics

In [ ]:
from ultralytics import YOLO

In [ ]:
import cv2
import numpy as np
import time
from ultralytics import YOLO
from PIL import Image

def process_tesla_vision(input_path, output_path, unet_model):
    # 1. Chargement de YOLOv8 Nano (pré-entraîné sur COCO : voitures, piétons, etc.)
    yolo_model = YOLO('yolov8n.pt')
    unet_model.eval()

    cap = cv2.VideoCapture(input_path)
    width  = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps_video = int(cap.get(cv2.CAP_PROP_FPS))

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(output_path, fourcc, fps_video, (width, height))

    # Transformation pour ton U-Net
    unet_transform = T.Compose([
        T.Resize((384, 640)),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    print("Segmentation + Détection ...")

    while cap.isOpened():
        start_time = time.time()
        ret, frame = cap.read()
        if not ret: break

        # --- PARTIE A : SEGMENTATION (TON MODÈLE) ---
        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(img_rgb)
        unet_input = unet_transform(pil_img).unsqueeze(0).to(DEVICE)

        with torch.no_grad():
            unet_output = unet_model(unet_input)
            pred = torch.sigmoid(unet_output).cpu().numpy()[0][0]
            mask = (pred > 0.5).astype(np.uint8)

        # Appliquer le tapis bleu
        mask_hd = cv2.resize(mask, (width, height), interpolation=cv2.INTER_NEAREST)
        blue_carpet = frame.copy()
        blue_carpet[mask_hd == 1] = [255, 100, 0] # Bleu
        result_frame = cv2.addWeighted(blue_carpet, 0.4, frame, 0.6, 0)

        # --- PARTIE B : DÉTECTION (YOLOv8) ---
        # On ne garde que les classes qui nous intéressent (voiture, bus, camion, piéton, vélo)
        # IDs COCO : 0: piéton, 1: vélo, 2: voiture, 3: moto, 5: bus, 7: camion
        yolo_results = yolo_model(frame, device=DEVICE, verbose=False, classes=[0, 1, 2, 3, 5, 7])[0]

        # Dessiner les boîtes de YOLO sur le résultat de la segmentation
        for box in yolo_results.boxes:
            x1, y1, x2, y2 = map(int, box.xyxy[0])
            cls = int(box.cls[0])
            conf = float(box.conf[0])
            label = f"{yolo_model.names[cls]} {conf:.2f}"

            # Dessiner la boîte
            cv2.rectangle(result_frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(result_frame, label, (x1, y1 - 10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)

        # --- PARTIE C : FPS & FINAL ---
        fps_current = 1 / (time.time() - start_time)
        cv2.putText(result_frame, f"FPS: {fps_current:.1f}", (50, 50),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 2)

        out.write(result_frame)

    cap.release()
    out.release()
    print(f"✅ Vidéo terminée ! Sauvegardée sous : {output_path}")

# Lancement
process_tesla_vision("13605989_1920_1080_30fps.mp4", "tesla_full_vision4.mp4", model)

🚀 Tesla Vision en cours (Segmentation + Détection)...
✅ Vidéo terminée ! Sauvegardée sous : tesla_full_vision4.mp4
